In [28]:
import pandas as pd
import numpy as np
import time
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
#Create a function for RFE feature selection with each algorithms.
def rfeFeature(indep_X, dep_Y, n):
    rfe_feature_list = []
    rfe_selected_names = []
    #1.logistic regression
    from sklearn.linear_model import LogisticRegression
    log_model = LogisticRegression(solver='lbfgs')
    #2.Random Forest
    from sklearn.ensemble import RandomForestClassifier
    RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    #3.Decision Tree
    from sklearn.tree import DecisionTreeClassifier
    DT = DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
    #4.SVM model
    from sklearn.svm import SVC
    svc_model = SVC(kernel = 'linear', random_state = 0)
    rfe_model_list = [log_model, RF, DT, svc_model]
    #let's loop through rfe model list (each model one by one) and get the selected features
    for i in rfe_model_list:
        from sklearn.feature_selection import RFE
        rferr = RFE(i, n)
        print (i)
        fit1 = rferr.fit(indep_X, dep_Y) # to select features from input as per output
        # Selected feature names
        mask = fit1.support_
        selected_cols = indep_X.columns[mask]
        #print("Selected features:", list(selected_cols))
        
        # Store both transformed data and names
        rfe_feature_list.append(fit1.transform(indep_X))
        rfe_selected_names.append(selected_cols)
    
    return rfe_feature_list, rfe_selected_names

In [3]:
#train test split
def split_scaler(indep_X, dep_Y):
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size=0.25, random_state=0)
    
    # to standardise the data
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)
    
    return X_train, X_test, y_train, y_test

In [4]:
#Metrics
def cm_prediction(classifier, X_test):
    y_pred = classifier.predict(X_test)
    
    #1 confusion matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, y_pred)
    #2 Accuracy
    from sklearn.metrics import accuracy_score
    Accuracy = accuracy_score(y_test, y_pred)
    #3 classification report
    from sklearn.metrics import classification_report
    report = classification_report(y_test, y_pred)
    
    # We will just return Accuracy. We can return the other matrics if needed.
    return Accuracy

In [5]:
#creation of models.

def logistic(X_train,y_train,X_test):       
        from sklearn.linear_model import LogisticRegression
        classifier = LogisticRegression(random_state = 0)
        classifier.fit(X_train, y_train)
        Accuracy=cm_prediction(classifier,X_test)
        return Accuracy      
    
def svm_linear(X_train,y_train,X_test):                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(X_train, y_train)
        Accuracy=cm_prediction(classifier,X_test)
        return Accuracy
    
def svm_NL(X_train,y_train,X_test):                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(X_train, y_train)
        Accuracy=cm_prediction(classifier,X_test)
        return Accuracy

def Navie(X_train,y_train,X_test):       
        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(X_train, y_train)
        Accuracy=cm_prediction(classifier,X_test)
        return Accuracy    
    
def knn(X_train,y_train,X_test):
        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(X_train, y_train)
        Accuracy=cm_prediction(classifier,X_test)
        return Accuracy
    
def Decision(X_train,y_train,X_test):
        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        Accuracy=cm_prediction(classifier,X_test)
        return Accuracy

def random(X_train,y_train,X_test):
        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        Accuracy=cm_prediction(classifier,X_test)
        return Accuracy

In [6]:
#Get all accuracy into a dataframe

def rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf):
    rfedataframe=pd.DataFrame(index=['Logistic','SVC','Random','DecisionTree'],columns=['Logistic','SVMl','SVMnl','KNN','Navie','Decision','Random'])
    for number,idex in enumerate(rfedataframe.index):
        rfedataframe['Logistic'][idex]=acclog[number]       
        rfedataframe['SVMl'][idex]=accsvml[number]
        rfedataframe['SVMnl'][idex]=accsvmnl[number]
        rfedataframe['KNN'][idex]=accknn[number]
        rfedataframe['Navie'][idex]=accnav[number]
        rfedataframe['Decision'][idex]=accdes[number]
        rfedataframe['Random'][idex]=accrf[number]
    return rfedataframe

In [7]:
dataset = pd.read_csv("prep.csv",index_col=None)
df2 = dataset
df2 = pd.get_dummies(df2, drop_first=True)

In [8]:
indep_X = df2.drop('classification_yes', 1) # droping output
dep_Y=df2['classification_yes'] # keep only op

C:\Anaconda3\envs\aiml\lib\site-packages\ipykernel_launcher.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  """Entry point for launching an IPython kernel.


In [9]:
rfe_feature_list,selected_cols = rfeFeature(indep_X, dep_Y, 3)


C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\decomposition\online_lda.py:29: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  EPS = np.finfo(np.float).eps
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\ensemble\gradient_boosting.py:32: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  from ._gradient_boosting import predict_stages
C:\Anaconda3\

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='warn', n_jobs=None, penalty='l2',
                   random_state=None, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)


C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\linear_model\logistic.py:947: ConvergenceWarning: lbfgs failed to converge. Increase the number of iterations.
  "of iterations.", ConvergenceWarning)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'):
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\linear_model\logistic.py:947: ConvergenceWarning: lbfgs failed to converge. Increase the number of iterations.
  "of iterations.", ConvergenceWarning)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'):
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\linear_model\logistic.py:947: ConvergenceWarning: lbfgs failed to converge. Increase the number of iterations.
  "of iterations

RandomForestClassifier(bootstrap=True, class_weight=None, criterion='entropy',
                       max_depth=None, max_features='auto', max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=10,
                       n_jobs=None, oob_score=False, random_state=0, verbose=0,
                       warm_start=False)


C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\tree\tree.py:163: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  y_encoded = np.zeros(y.shape, dtype=np.int)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\tree\tree.py:163: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the rele

C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'):
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\ensemble\forest.py:489: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  y_store_unique_indices = np.zeros(y.shape, dtype=np.int)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  

C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\tree\tree.py:163: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  y_encoded = np.zeros(y.shape, dtype=np.int)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\tree\tree.py:163: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the rele

DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
                       max_features='sqrt', max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, presort=False,
                       random_state=0, splitter='best')
SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='auto_deprecated',
    kernel='linear', max_iter=-1, probability=False, random_state=0,
    shrinking=True, tol=0.001, verbose=False)


In [11]:
#create eempty list to store accuracy score
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

In [12]:
for i in rfe_feature_list: # loop through each set of selected features from each RFE algorithms
    
    #test train split
    X_train, X_test, y_train, y_test = split_scaler(i, dep_Y)
    
    Accuracy = logistic(X_train,y_train,X_test)
    acclog.append(Accuracy)
    
    Accuracy = svm_linear(X_train,y_train,X_test)
    accsvml.append(Accuracy)
    
    Accuracy = svm_NL(X_train,y_train,X_test)
    accsvmnl.append(Accuracy)
    
    Accuracy = Navie(X_train,y_train,X_test)     
    accnav.append(Accuracy)  
    
    Accuracy = knn(X_train,y_train,X_test)
    accknn.append(Accuracy)
    
    Accuracy = Decision(X_train,y_train,X_test)
    accdes.append(Accuracy)
        
    Accuracy = random(X_train,y_train,X_test)
    accrf.append(Accuracy)

C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\linear_model\base.py:291: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  indices = (scores > 0).astype(np.int)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\neighbors\base.py:908: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itsel

C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\neighbors\base.py:908: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  self._y = np.empty(y.shape, dtype=np.int)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\neighbors\base.py:441: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  old_joblib = LooseVersion(joblib_version) < LooseVersion('0.12')
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\neighbors\base.py:441: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version inst

In [13]:
len(acclog)

4

In [14]:
result = rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)
result

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
Logistic,0.94,0.94,0.94,0.94,0.94,0.94,0.94
SVC,0.94,0.94,0.94,0.94,0.9,0.91,0.92
Random,0.97,0.98,0.98,0.98,0.79,0.97,0.97
DecisionTree,0.87,0.87,0.87,0.87,0.87,0.87,0.87


In [41]:
#let's go with features selected by RF and model created by SVML
final_seleceted_feat = rfe_feature_list[1]

X_train, X_test, y_train, y_test = train_test_split(final_seleceted_feat, dep_Y, test_size=0.25, random_state=0)
    
# to standardise the data    
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

#save the sacled model for delpyment stage.
import joblib
joblib.dump(sc, "scaler.pkl")


#not required to standardize the o/p in classification as it is not continuous numerical value
classifier = SVC(kernel = 'linear', random_state = 0)
classifier.fit(X_train, y_train)

SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='auto_deprecated',
    kernel='linear', max_iter=-1, probability=False, random_state=0,
    shrinking=True, tol=0.001, verbose=False)

In [15]:
selected_cols

[Index(['sg_c', 'sg_d', 'htn_yes'], dtype='object'),
 Index(['sc', 'hrmo', 'pcv'], dtype='object'),
 Index(['hrmo', 'sg_c', 'dm_yes'], dtype='object'),
 Index(['sg_d', 'dm_yes', 'appet_yes'], dtype='object')]

In [35]:
import pickle
filename="finalized_model_linear.sav"
pickle.dump(classifier,open(filename, 'wb'))

#from joblib we case use the below code as well to save the model as alternative for pickle
#joblib.dump(classifier, "finalized_model_linear.pkl")
#loaded_model = joblib.load("model.pkl")


In [42]:
loaded_model = pickle.load(open("finalized_model_linear.sav", 'rb'))
# RF has selected these columns [sc', 'hrmo', 'pcv]

scaled_model = joblib.load("scaler.pkl")

preinput = scaled_model.transform([[0.7, 10.7, 34]])

f_result = loaded_model.predict(preinput)
f_result

array([1], dtype=uint8)